# اليوم الرابع — مختبر 7: تحسين الاستدلال والخدمة
## Day 4 — Lab 7: Project-Artifact Optimisation & Serving

**Program:** SDA-AIE-211 — Natural Language Processing with Transformers  
**Instructor:** Meaad Al-Marri | ميعاد المري  
**Academy:** SDAIA Academy  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yousef0197/sda_nlp-bayan-capstone/blob/main/notebooks/08_optimization_serving.ipynb)

هذا الدفتر مخصص لبوابة **Gate D** على نموذج بيان الفعلي. مسار `SYSTEMS_SMOKE` السابق يبقى دليلًا منفصلًا على آلية التصدير، لكنه لا يُستخدم هنا كدليل المشروع النهائي.


## عقد القياس النهائي | Final measurement contract

القياس النهائي يربط **نفس** نموذج تصنيف الموضوع ثنائي اللغة مع إصدار المعالجة، خريطة الفئات، workload validation العربي/الإنجليزي، ومرشحي PyTorch/ONNX/INT8. الأوزان الكبيرة تُنشأ مؤقتًا ولا تُرفع إلى GitHub؛ التقرير الصغير فقط يُحفظ في `reports/t10_project_artifact_benchmark.json`.

الميزانية أدناه معلنة قبل قياس المرشحين. الجودة تقاس بـ `Macro-F1` على نفس validation workload، و`quality tax` هو مقدار الانخفاض مقارنةً بـ PyTorch FP32.


In [ ]:
# ===== Student Gate D configuration =====
PROJECT_MODE = True
PROJECT_MODEL_SOURCE = "/content/bayan_day4_artifacts/project_model"
PROJECT_TOKENIZER_SOURCE = PROJECT_MODEL_SOURCE
PROJECT_VALIDATION_CSV = "data/sample/bayan_day4_validation.csv"
PROJECT_PREPROCESSING_VERSION = "ar-en-v1"
BUDGET_PROVENANCE = "STUDENT_DEFINED_BEFORE_MEASUREMENT"
ARTEFACT_ROLE = "PROJECT_ARTIFACT"
RESULT_LABEL = "MEASURED"
PERFORMANCE_BUDGET = {
    "max_p95_ms": 1000.0,
    "min_throughput_items_s": 0.1,
    "max_quality_tax": 0.05,
    "target_device": "cpu",
}
assert PROJECT_MODE is True
assert BUDGET_PROVENANCE == "STUDENT_DEFINED_BEFORE_MEASUREMENT"
print("ARTEFACT_ROLE", ARTEFACT_ROLE)
print("BUDGET_PROVENANCE", BUDGET_PROVENANCE)
print("TARGET", PERFORMANCE_BUDGET)


In [ ]:
# Reproduce from a clean Colab runtime.
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Yousef0197/sda_nlp-bayan-capstone.git"
REPO_DIR = Path("/content/bayan_project")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements-day4.txt"])
env = os.environ.copy()
env["PYTHONPATH"] = "src"
env["HF_HUB_DISABLE_TELEMETRY"] = "1"
env["DO_NOT_TRACK"] = "1"
env["ORT_DISABLE_TELEMETRY"] = "1"
subprocess.check_call([sys.executable, "scripts/run_gate_d_project_artifact.py"], env=env)
print("GATE_D_REPRODUCTION_RUN=PASS")


In [ ]:
from pathlib import Path
import json

REPORT_PATH = Path("reports/t10_project_artifact_benchmark.json")
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
assert report["gate_d_status"] == "PASS"
assert report["artefact_role"] == "PROJECT_ARTIFACT"
assert report["result_label"] == "MEASURED"
assert report["budget_provenance"] == "STUDENT_DEFINED_BEFORE_MEASUREMENT"
assert report["workload"]["arabic_count"] > 0
assert report["workload"]["english_count"] > 0
assert report["quality"]["quality_tax"] <= report["budget"]["max_quality_tax"]
assert report["budget_assessment"]["status"] == "PASS"
assert all(item["status"] == "PASS" for item in report["canaries"])
print(json.dumps({
    "gate_d_status": report["gate_d_status"],
    "selected_runtime": report["selected_runtime"]["name"],
    "quality": report["quality"],
    "p95_ms": report["candidates"][report["selected_runtime"]["name"]]["benchmark"]["p95_ms"],
}, ensure_ascii=False, indent=2))
print("DAY4_NOTEBOOK8_CORE=PASS")


## Evidence boundary

- `reports/t10_project_artifact_benchmark.json` = **Gate D PROJECT_ARTIFACT** measurement.
- `reports/t10_local_cpu_http_benchmark.json` = separate real HTTP/Uvicorn CPU measurement at concurrency 16.
- Older `SYSTEMS_SMOKE` measurements demonstrate export/optimisation mechanics only and are not substituted for the project-artifact evidence.
- Rollback remains the tuned PyTorch FP32 checkpoint if an optimised candidate violates latency, throughput, parity, or quality budget.
